In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# 2. Load data
df = pd.read_csv("SwedishMotorInsurance.csv")
print(df.head())
print(df.shape)

# 3. Handle missing values
df = df.dropna()

# 4. Separate numerical and text columns
num_cols = df.select_dtypes(include=np.number).columns
text_cols = df.select_dtypes(include="object").columns

# 5. Numerical features
X_num = StandardScaler().fit_transform(df[num_cols])

# 6. NLP features
if len(text_cols) > 0:
    text = df[text_cols].astype(str).agg(" ".join, axis=1)
    tfidf = TfidfVectorizer(stop_words="english", max_features=500)
    X_text = tfidf.fit_transform(text).toarray()
else:
    X_text = np.empty((len(df), 0))

# 7. Combine features
X = np.hstack((X_num, X_text))

# 8. Find best K using silhouette score
scores = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores.append(silhouette_score(X, labels))

best_k = range(2, 7)[np.argmax(scores)]
print("Best K:", best_k)

# 9. K-Means clustering
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)

# 10. Cluster results
print("\nCluster counts:")
print(df["Cluster"].value_counts())

# 11. PCA visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df["Cluster"], cmap="viridis")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Swedish Motor Insurance - K-Means Clusters")
plt.colorbar(label="Cluster")
plt.show()

# 12. Cluster summary
print("\nCluster Summary:")
print(df.groupby("Cluster")[list(num_cols)].mean())

# 13. Save results
df.to_csv("SwedishMotorInsurance_clustered.csv", index=False)
print("\nSaved as SwedishMotorInsurance_clustered.csv")